# Data Preprocessing
This notebook starts with downloading the dataset and then performs data preprocessing steps to create a clean dataset for further analysis.

## Imports

In [ ]:
import os
import zipfile
import requests
from tqdm import tqdm
import shutil
import csv
import re
import json
from collections import Counter, defaultdict

## Downloading the Dataset and extracting it

This ran more than 30 min even with really fast internet I think they are throttling the download speed.

In [ ]:
output_dir = "data/dta_complete_2021-10-19"
os.makedirs(output_dir, exist_ok=True)
url = "https://www.deutschestextarchiv.de/media/download/dta_komplett_2021-10-19_tcf.zip"

# Download
with requests.get(url, stream=True) as r, open("data/dta.zip", "wb") as f:
    total_size = int(r.headers.get('content-length', 0))
    with tqdm(total=total_size, unit='B', unit_scale=True, desc="Downloading") as t:
        for chunk in r.iter_content(1024):
            f.write(chunk)
            t.update(len(chunk))
print("\nDownload completed. Extracting files...")

# Extract
with zipfile.ZipFile("data/dta.zip", "r") as zip_ref:
    zip_ref.extractall(output_dir)
print("Extraction completed. Processing files...")

# Cleanup and rearrange
shutil.rmtree(os.path.join(output_dir, "dta_komplett_2021-10-19", "simple"), ignore_errors=True)
full_path = os.path.join(output_dir, "dta_komplett_2021-10-19", "full")
for file in os.listdir(full_path):
    shutil.move(os.path.join(full_path, file), output_dir)
print("Files processed and moved to the output directory.")

# Remove leftover dirs
shutil.rmtree(os.path.join(output_dir, "dta_komplett_2021-10-19"), ignore_errors=True)
os.remove("data/dta.zip")
print("Cleanup completed. The dataset is ready for use in the output directory:", output_dir)

Downloading:   4%|▍         | 177M/4.62G [01:38<53:37, 1.38MB/s]   

## Functions for Data Preprocessing

In [ ]:
def extract_sentences_from_text(text):
    """
    Extrahiere rohe XML-Fragmente, die aus <token>…</token>-Elementen bestehen
    und mit einem Punkt-Token enden.
    """
    # Pattern für ein ganzes Satzfragment (eine Reihe von Token-Elementen, endend auf Punkt)
    sentence_re = re.compile(
        r'(?:<token\b[^>]*>.*?</token>\s*)+?<token\b[^>]*>\.</token>',
        flags=re.DOTALL | re.IGNORECASE
    )
    return [m.group(0).strip() for m in sentence_re.finditer(text)]

def process_files(input_dir, keyword_level, examples_per_keyword):
    """
    Durchsuche alle .xml/.tcf, match Keywords in den rohen XML-Sätzen,
    sammle bis zu examples_per_keyword Beispiele und zähle Frequenzen.
    """
    examples = defaultdict(list)  # kw -> list of dicts {raw_xml, file, sentence_id}
    freq     = Counter()
    # Precompile Regex für Keywords (aber weiter über raw text matchen)
    patterns = {
        kw: re.compile(rf'\b{re.escape(kw)}\b', flags=re.IGNORECASE)
        for kw in keyword_level
    }

    for root, _, files in os.walk(input_dir):
        for fname in files:
            if not fname.lower().endswith(('.xml', '.tcf')):
                continue
            path = os.path.join(root, fname)
            try:
                raw = open(path, encoding='utf-8').read()
            except Exception as e:
                print(f"Warning: {fname}: {e}")
                continue

            # Für jedes extrahierte Satz-Fragment
            for sid, raw_xml in enumerate(extract_sentences_from_text(raw)):
                for kw, pat in patterns.items():
                    if pat.search(raw_xml):
                        freq[kw] += 1
                        if len(examples[kw]) < examples_per_keyword:
                            examples[kw].append({
                                "raw_xml": raw_xml,
                                "file": fname,
                                "sentence_id": sid
                            })
                # Abbruch, wenn alle Limits erreicht
                if all(len(examples[k]) >= examples_per_keyword for k in keyword_level):
                    return examples, freq

    return examples, freq

def write_output_json(output_path, examples, freq, keyword_level):
    """
    Schreibe ein JSON-Array mit allen Matches:
    [
      {
        "file": "...",
        "sentence_id": 42,
        "raw_xml": "<token ...>…</token>…",
        "keyword": "je",
        "score": 4,
        "frequency": 21
      },
      …
    ]
    """
    out = []
    for kw, hits in examples.items():
        score = keyword_level.get(kw, "")
        count = freq.get(kw, 0)
        for hit in hits:
            out.append({
                "file": hit["file"],
                "sentence_id": hit["sentence_id"],
                "raw_xml": hit["raw_xml"],
                "keyword": kw,
                "score": score,
                "frequency": count
            })

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f"✅ JSON-Output geschrieben nach {output_path}")

def load_testset(testset_path):
    """
    Load keywords and their scores from a CSV with columns 'Keyword','Score'.

    Returns:
        dict: mapping keyword -> score
    """
    keyword_level = {}
    with open(testset_path, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            kw = row.get("Keyword", "").strip()
            score = row.get("Score", "").strip()
            if kw:
                keyword_level[kw] = score
    return keyword_level


## Running the Preprocessing

In [ ]:
input_dir            = "test_data"
testset_path         = "testset.csv"
output_path_json     = "test_data/annotated_data.json"
examples_per_keyword = 2

keyword_level = load_testset(testset_path)
examples, freq = process_files(input_dir, keyword_level, examples_per_keyword)
write_output_json(output_path_json, examples, freq, keyword_level)